# 05. Clasificación asistida con RoBERTuito

Se generan propuestas automáticas con los textos de la planilla humana. Se cotejan
los identificadores, la fuente y el texto original con la salida del cuaderno 04.
Las propuestas se guardan en una columna separada de la etiqueta humana. La planilla
anotada se copia byte por byte para el cuaderno 08.

Si abres este cuaderno directamente en Colab, ejecuta su primera celda y autoriza Google Drive cuando se solicite. Para recorrer toda la secuencia de una vez, usa `INICIAR_EN_COLAB.ipynb`.

**Salidas guardadas.** Las tablas, figuras y registros visibles proceden de la corrida original `corrida_03_epocas`. Esta copia incorpora las preguntas de carga de 06–08. Las salidas y los números de ejecución conservan su origen histórico. Abre el cuaderno y desplázate para exponer los resultados.

`GENERAR_PROPUESTAS = False` reutiliza la clasificación asistida guardada al ejecutar todas las celdas. Las etiquetas humanas se leen de la planilla original.

La copia pública presenta el código de esta etapa. Sus salidas por publicación se conservan en la copia de Drive.

In [ ]:
# Inicio al abrir este cuaderno directamente en Google Colab.
from pathlib import Path

if not Path("corrida.json").is_file():
    import os
    import subprocess
    import sys

    try:
        from google.colab import drive
    except ImportError as error:
        raise FileNotFoundError("En Jupyter, abre la carpeta que contiene corrida.json.") from error

    proyecto_drive = Path("/content/drive/MyDrive/Tesis_RoBERTuito_M0_M1_M2")
    if not (proyecto_drive / "corrida.json").is_file():
        drive.mount("/content/drive", timeout_ms=180000)
    if not (proyecto_drive / "corrida.json").is_file():
        raise FileNotFoundError("Busca Tesis_RoBERTuito_M0_M1_M2 directamente en Mi unidad.")

    os.chdir(proyecto_drive)
    sys.path.insert(0, str(proyecto_drive))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(proyecto_drive / "requirements-cuadernos.txt")],
        check=True,
    )
    print("Carpeta de la tesis conectada:", proyecto_drive)

from rutas import carpeta_salidas

from pathlib import Path

RAIZ = Path.cwd()
if not (RAIZ / "corrida.json").is_file():
    raise FileNotFoundError("Abre Jupyter en la carpeta que contiene corrida.json.")

from shutil import copyfile

import pandas as pd
import json
from hashlib import file_digest

GENERAR_PROPUESTAS = False  # @param {type:"boolean"}
tablas = carpeta_salidas(RAIZ) / "TABLAS"
entrada = tablas / "04_preprocesamiento/datos_preprocesados_v4.xlsx"
oficial = RAIZ / "TABLAS/06_clasificacion/clasificacion_asistida_v5_etiquetada.xlsx"
referencia = pd.read_excel(oficial, sheet_name="clasificacion", index_col="id", keep_default_na=False)
if not referencia.index.is_unique:
    raise ValueError("Hay identificadores repetidos en la planilla humana.")
if GENERAR_PROPUESTAS:
    import torch
    from entrenamiento import cargar_modelo
    from evaluar import predecir

    columnas = ["fuente", "texto_real"]
    preprocesados = pd.read_excel(entrada, sheet_name="datos", usecols=["id", *columnas],
                                 index_col="id", keep_default_na=False)
    if not preprocesados.index.is_unique or not preprocesados.reindex(referencia.index).equals(referencia[columnas]):
        raise ValueError("Las publicaciones del cuaderno 04 difieren de la planilla humana.")
    dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    modelo, tokenizer = cargar_modelo(RAIZ / "modelos/corrida_03_epocas/M0")
    modelo.to(dispositivo).eval()
else:
    guardadas = RAIZ / "insumos/propuestas_reproducidas.xlsx"
    registro = json.loads((guardadas.with_suffix(".json")).read_text(encoding="utf-8"))
    for archivo, esperado in (
        (guardadas, registro["propuestas_sha256"]),
        (oficial, registro["anotacion_sha256"]),
        (RAIZ / "modelos/corrida_03_epocas/M0/model.safetensors", registro["pesos_sha256"]),
    ):
        with archivo.open("rb") as contenido:
            if file_digest(contenido, "sha256").hexdigest() != esperado:
                raise ValueError("Las propuestas guardadas no corresponden a los archivos de esta copia.")
    propuestas = pd.read_excel(guardadas, index_col="id", keep_default_na=False)
    if not propuestas.index.is_unique or not propuestas[referencia.columns].equals(referencia):
        raise ValueError("Las propuestas guardadas difieren de la planilla humana.")
    print("Se reutilizan las propuestas de clasificación guardadas.")

In [ ]:
if GENERAR_PROPUESTAS:
    predicciones = predecir(referencia["texto_procesado"], modelo, tokenizer, dispositivo)
    propuestas = referencia.copy()
    propuestas["polaridad_sugerida_reproducida"] = [clase for clase, _ in predicciones]

carpeta = tablas / "05_clasificacion"
carpeta.mkdir(parents=True, exist_ok=True)
salida_propuestas = carpeta / "propuestas_reproducidas.xlsx"
if GENERAR_PROPUESTAS:
    propuestas.reset_index().to_excel(salida_propuestas, index=False)
elif guardadas.resolve() != salida_propuestas.resolve():
    copyfile(guardadas, salida_propuestas)
if oficial.resolve() != (carpeta / oficial.name).resolve():
    copyfile(oficial, carpeta / oficial.name)
print(f"Propuestas disponibles: {len(propuestas)}")
print("Etiquetas humanas conservadas:", propuestas["etiqueta_humana"].value_counts().to_dict())